# 2.1 算子数据流：三条流的编织物

装备清点完毕（1.2/1.3），现在把 AI Core 拆开，取**第一块积木：数据流**。

本节回答：一个算子在硬件上运行时，**指令、同步、数据**分别在怎么流动？这是读懂任何 Ascend C Kernel 的总纲——包括我们第3篇要写的 FA。

## 1. AI Core 抽象架构总图

<img src="./images/ai_core_abstract_arch.png" width="640">

> **看图要点**：AI Core 由三类组件构成——**计算单元**（Cube/Vector/Scalar）、**存储单元**（Local Memory 与 Global Memory，详见 2.2 节）、**搬运单元**（DMA，负责 GM ↔ Local 及 Local 各级之间）。这套抽象在 A2/A3/950 上一致，仅容量与计算能力不同——所以本章的六块积木“一次学习，三代通用”。

## 2. 多核视角：SPMD 与 block_idx

以上都是单个 AI Core 的内部图景。而一个 Kernel 被调用时，硬件上启动的是 **N 个并行运行实例（Block）**：

- 它们执行**同一份程序、同一组参数**，唯一的区别是内建变量 **block_idx** 的值不同（概念上类似线程 ID）；
- 算子用 block_idx 把待处理数据切分给各 Block——这就是 **SPMD（Single Program Multiple Data）**，Ascend C 的多核并行模型。

对 FA 而言，输入的 B×S（batch × sequence）维天然适合做 Block 级切分：每个 Block 认领一批（或多批）Query 头，各自在核内再做 Tiling 分块。第3篇的 FA Kernel 里，`GetBlockIdx()` 拿到的就是这把切分钥匙。

（后面 2.1~2.6 讲的“三条流、六块积木”，都是**单个 Block 内部**的故事。）

## 3. 三条流：指令流 / 同步流 / 数据流

Kernel 一旦启动，AI Core 内部同时上演三场戏：

| 流 | 主体 | 干什么 | FA 中的例子 |
|--|--|--|--|
| **异步指令流** | Scalar | 把 Vector/Cube/搬运指令发射到各单元的独立队列，各单元异步并行执行 | 循环体里依次发射 MTE2、mmad、向量指令 |
| **同步信号流** | Scalar | 在指令间插入依赖信号，保证“搬运完成才计算”的顺序 | Q 块搬入完成 → 才允许 QK^T 开始 |
| **计算数据流** | DMA + 计算单元 | 搬入 → 计算 → 搬出 | GM 的 Q/K/V → 片上 → S/P 中间量 → O 输出 |

> **为什么必须有三条流？** 因为各执行单元是**异步**的——Cube 不等 Vector，MTE 不等计算。没有同步流，Cube 可能在数据还没搬到位时就开算，读到的就是垃圾。同步的具体指令（SetFlag/WaitFlag）在 2.6 节展开。

## 4. 计算数据流：搬运-计算-搬运

最基础的 Vector 算子数据流是一个三段式循环：

<img src="./images/data_moving_compute_process.png" width="560">

> **看图要点**：**CopyIn**（MTE2 把数据从 GM 搬进 UB）→ **Compute**（Vector 在 UB 上计算）→ **CopyOut**（MTE3 把结果从 UB 搬回 GM）。片上存储放不下整个输入时，就把输入切块、循环执行这个三段式——这正是 FA 分块计算（第2章 Tiling）在硬件上的自然形态。

## 5. 矩阵算子的完整数据流：以 MatMul 为例

FA 的两大主力计算（QK^T、P·V）都是矩阵乘。一次矩阵乘的数据旅行要走完全部存储层级：

<img src="./images/matmul_compute_flow.png" width="640">

> **看图要点（建议在这张图上停留 3 分钟）**：
> 1. **GM → L1**（MTE2）：矩阵 A、B 分块后搬入片上 L1，此时是常规 ND 排布；
> 2. **L1 → L0A / L0B**（MTE1）：搬运同时把数据**排布成 Cube 要求的分形格式**（NZ 等，详见 2.2 节）；
> 3. **L0A/L0B → L0C**（mmad 指令）：Cube 沿 K 维循环累加，结果落 L0C；
> 4. **L0C → GM/UB**（MTE3/FIXPIPE）：结果写出，可顺带排布转换（ZN → ND）。

**四个存储层级、三条搬运指令、一条计算指令**——把这张图刻进脑子，第3篇写 FA 时你会不断与它重逢。

## 6. FA 算子的数据流映射

把上面两张图叠起来，就是 FlashAttention 的硬件形态：

<img src="./images/fa_hw_mapping.svg" width="700">

> **看图要点**：**两次 Cube 密集计算（①QK^T、③P·V）夹一段 Vector 密集计算（②Online Softmax）**。
> - ① S = Q·K^T：Cube 主场，输入走 L1→L0A/B，结果 S 落 L0C；
> - ② 行 Max / Exp / 行 Sum / Rescale：Vector 主场，S 转入 UB 后逐行处理；
> - ③ O = P·V：又是 Cube 主场，O 在 L0C 上累加，最终 FIXPIPE 输出。

在第2章你已推导过这三段的数学；从今天起，请用“硬件视角”重新审视它们——**FA 性能优化的全部空间，都藏在这三段与三条流的编排里**（第7~9章）。

## 小结

| 关键词 | 一句话 |
|--|--|
| SPMD | 一份代码 N 个 Block，block_idx 决定各认领哪份数据 |
| 三条流 | 指令流（发射）、同步流（依赖）、数据流（搬运-计算-搬运） |
| 三段式 | CopyIn → Compute → CopyOut 的循环 |
| MatMul 全景 | GM→L1→L0A/B→L0C→GM，四层存储三条搬运 |
| FA 形态 | Cube → Vector → Cube 的三明治结构 |

下一节 [2.2 内存结构](01.04_memory_structure.ipynb)：把“数据放在哪里”这件事一次讲透。

## 课后练习

1.（单选）MatMul 计算中，把数据从 L1 搬到 L0A/L0B 并转为分形排布的是哪条指令通路？（A. MTE2　B. MTE1　C. MTE3　D. FIXPIPE）

2.（判断）异步指令流中，Scalar 发射指令后各执行单元立即执行，因此不需要任何同步机制。

3.（单选）FA 中 Online Softmax 三步递推主要发生在哪个存储层级上？（A. GM　B. L1　C. L0C　D. UB）

> 完成后查看 [answer/01.03_answer.txt](answer/01.03_answer.txt) 核对答案。